<a href="https://colab.research.google.com/github/Zikai-Dou/dataflow-ai-labs/blob/main/Lab4_Profiling_Memory_Access.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Profiling Memory Access Patterns in LLMs

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Module**: 4, Memory Systems  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes


---

## Learning objectives
By the end of this lab you will be able to:
1. Capture allocation/deallocation with torch.profiler (profile_memory=True).
2. Identify the memory-intensive operators and the peak allocation during decode.
3. Compute effective memory bandwidth for the heaviest operators.
4. Predict where a scratchpad hierarchy (Spyre) helps vs a GPU cache, per bottleneck.

## Prerequisites
Labs 1-3 complete, plus the Module 4 lecture on memory hierarchies, the KV cache, and scratchpad vs cache.

## Working through this lab
1. Run the notebook cells in order.
2. Complete each learner task before opening its **Hint**.
3. If you get stuck, expand the **Hint** to view the full reference solution from the authoring notebook.
4. Run each **Self-check** cell after completing the corresponding task.



## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                          ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## 0 · Setup

In [ ]:
!pip -q install transformers==4.44.2

import gc, math
import torch
import pandas as pd
from torch.profiler import profile, ProfilerActivity, record_function
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEV)
if DEV == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


We profile **`distilgpt2`** during a short generation so the **KV cache** grows and shows up in the
memory timeline, the Module 4 protagonist. The same harness works on any causal LM.

In [ ]:
MODEL_NAME = 'distilgpt2'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEV).eval()

PROMPT = 'Memory placement on a dataflow accelerator is decided by the compiler, because'
ids = tok(PROMPT, return_tensors='pt').input_ids.to(DEV)
print('prompt tokens:', ids.shape[-1])

---
## 1 · Warm-up: a short generation
Generate a few tokens so kernels are compiled and the KV cache is populated before profiling.

In [ ]:
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=16, do_sample=False, pad_token_id=tok.eos_token_id)
print('generated tokens:', out.shape[-1] - ids.shape[-1])
if DEV == 'cuda': torch.cuda.synchronize()

---
## 2 · Exercise: Profile inference with memory tracking
Implement `profile_generation`: run a `max_new_tokens`-token generation inside a `torch.profiler.profile`
context with `profile_memory=True` and `record_shapes=True`, recording the right activities for the device.
Return the `prof` object.

> Hint: `activities=[ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if cuda else [])`.

In [ ]:
# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
def profile_generation(model, ids, new_tokens=48):
    """Profile a short generation; return the profiler object."""
    acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if ids.is_cuda else [])
    with profile(activities=acts, profile_memory=True, record_shapes=True) as prof:
        with torch.no_grad():
            with record_function('generate'):
                model.generate(ids, max_new_tokens=new_tokens, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        if ids.is_cuda: torch.cuda.synchronize()
    return prof

prof = profile_generation(model, ids, new_tokens=48)
print('profiling complete')
```

</details>


---
## 3 · Exercise: Peak allocation and memory-intensive operators
Implement `memory_table`: from `prof.key_averages()`, build a DataFrame of the top operators by memory
usage (use `self_device_memory_usage` on GPU, else `self_cpu_memory_usage`), with their total time. Return
the top `k`. Also report **peak** allocation (`torch.cuda.max_memory_allocated` on GPU).

In [ ]:
# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
def memory_table(prof, k=8, cuda=True):
    """Return a DataFrame of the top-k operators by memory usage."""
    rows = []
    for e in prof.key_averages():
        if cuda:
            mem = getattr(e, 'self_device_memory_usage', 0) or getattr(e, 'self_cuda_memory_usage', 0)
        else:
            mem = e.self_cpu_memory_usage
        t_us = getattr(e, 'device_time_total', 0) or getattr(e, 'cuda_time_total', 0) if cuda else e.cpu_time_total
        rows.append({'op': e.key, 'self_mem_MB': mem / 1e6,
                     'time_ms': t_us / 1e3, 'calls': e.count})
    df = pd.DataFrame(rows)
    df = df[df['self_mem_MB'].abs() > 0].sort_values('self_mem_MB', ascending=False)
    return df.head(k).reset_index(drop=True)

cuda = (DEV == 'cuda')
top = memory_table(prof, k=8, cuda=cuda)
if cuda:
    print(f'peak allocated: {torch.cuda.max_memory_allocated()/1e6:.1f} MB')
top
```

</details>


---
## 4 · Exercise: Allocator activity vs. true memory bandwidth

The PyTorch profiler reports `self_cuda_memory_usage` (or `self_cpu_memory_usage`), which measures
**net allocator activity**: bytes allocated minus bytes freed; attributed to each operator.

**This is NOT memory bandwidth.** Net allocation can be zero for an in-place op, negative for ops
that free temporaries, and does not count reads at all. Dividing it by time gives "allocator activity
per second," not effective hardware bandwidth.

### Your task:
1. Compute "allocator activity / time" from your profiler table (to see why it's unreliable).
2. Then run a **controlled benchmark** where bytes transferred are analytically known: a `torch.copy_`
   of a known-size tensor. Compare the two numbers and explain in your analysis why they differ.

> The Spyre track (optional) shows how a scratchpad-based architecture makes traffic explicit.


### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead


In [ ]:
# Utility: robust timing with statistics
import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds."""
    # Warmup
    for _ in range(n_warmup):
        fn()
    if sync_cuda and torch.cuda.is_available():
        torch.cuda.synchronize()

    # Measure
    times = []
    for _ in range(n_measure):
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    return {
        "median_ms": float(np.median(times)),
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "p5_ms": float(np.percentile(times, 5)),
        "p95_ms": float(np.percentile(times, 95)),
        "iqr_ms": float(np.percentile(times, 75) - np.percentile(times, 25)),
        "n": n_measure,
    }

# Example usage
if torch.cuda.is_available():
    x = torch.randn(1024, 1024, device=DEVICE)
    stats = bench_robust(lambda: torch.mm(x, x))
    print(f"Matrix multiply (1024x1024) on {DEVICE}:")
    print(f"  Median: {stats['median_ms']:.3f} ms")
    print(f"  P5-P95: [{stats['p5_ms']:.3f}, {stats['p95_ms']:.3f}] ms")
    print(f"  IQR:    {stats['iqr_ms']:.3f} ms (over {stats['n']} runs)")
else:
    print("Skipping bench_robust demo (no GPU)")


In [ ]:
# 4.1: Allocator activity per time (NOT true bandwidth)
# This shows why profiler memory fields are unreliable bandwidth proxies.
# 4.2: Controlled bandwidth benchmark using analytically known byte traffic

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 4.1: Allocator activity per time (NOT true bandwidth)
# This shows why profiler memory fields are unreliable bandwidth proxies.
bw_proxy = top.copy()
bw_proxy['alloc_activity_GBps'] = bw_proxy.apply(
    lambda r: abs(r['self_mem_MB']) * 1e6 / 1e9 / (r['time_ms'] / 1e3)
    if r['time_ms'] > 0 else float('nan'), axis=1)
print("Allocator activity / time (NOT effective bandwidth):")
print(bw_proxy[['op', 'self_mem_MB', 'time_ms', 'alloc_activity_GBps']].head(5).to_string(index=False))
print()
print("⚠  These numbers are NOT hardware bandwidth; they measure net allocation, not traffic.")

# 4.2: Controlled bandwidth benchmark using analytically known byte traffic
# A copy of N bytes reads N and writes N → total traffic = 2N bytes.
import time

def measure_copy_bandwidth(size_mb=64, n_trials=20, device='cpu'):
    """Measure effective bandwidth using a controlled tensor copy.
    
    Known traffic: copy reads src (size_mb) + writes dst (size_mb) = 2 * size_mb.
    """
    nbytes = int(size_mb * 1e6)
    src = torch.randn(nbytes // 4, device=device)  # float32 = 4 bytes each
    dst = torch.empty_like(src)
    
    # Warm up
    for _ in range(3):
        dst.copy_(src)
    if device == 'cuda':
        torch.cuda.synchronize()
    
    # Timed trials
    times = []
    for _ in range(n_trials):
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        dst.copy_(src)
        if device == 'cuda':
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    
    median_s = sorted(times)[len(times) // 2]
    total_bytes = 2 * nbytes  # read + write
    bw_gbs = (total_bytes / 1e9) / median_s
    return bw_gbs, median_s * 1e3

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
copy_bw_gbs, copy_time_ms = measure_copy_bandwidth(size_mb=64, device=dev)
print(f"\nControlled copy benchmark ({dev}, 64 MB):")
print(f"  Median time:          {copy_time_ms:.3f} ms")
print(f"  Effective bandwidth:  {copy_bw_gbs:.1f} GB/s")
print(f"  (This IS effective bandwidth; traffic = 2×64 MB = 128 MB, analytically known)")
```

</details>


In [ ]:
# distilgpt2 config (used by the KV-cache formula in the Part 5 short answer)
N_LAYERS  = model.config.n_layer      # 6
N_HEADS   = model.config.n_head       # 12
HEAD_DIM  = model.config.n_embd // model.config.n_head   # 64
BATCH     = 1
BYTES_PER = 2   # FP16
kv_bytes = lambda seq: 2 * N_LAYERS * N_HEADS * HEAD_DIM * seq * BATCH * BYTES_PER
print(f'KV cache bytes/token = {kv_bytes(1):,} (2*L*H*d*batch*bytes)')


---
## 5 · The KV cache, measured (provided + short answer)
The cell below grows the generation length and records peak memory at each, isolating the KV cache's
linear growth. Run it, then answer the questions below.

In [ ]:
def peak_for_length(model, ids, n):
    if DEV == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model.generate(ids, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
    if DEV == 'cuda':
        torch.cuda.synchronize(); return torch.cuda.max_memory_allocated() / 1e6
    return float('nan')

lengths = [16, 32, 64, 128]
peaks = [peak_for_length(model, ids, n) for n in lengths]
pd.DataFrame({'new_tokens': lengths, 'peak_MB': [round(p, 1) for p in peaks]})

**Short answer (write below):**
1. How does peak memory scale as you double the number of generated tokens? Is that consistent with the
   KV-cache size formula `2 × layers × heads × head_dim × seq_len × batch × bytes`?
2. On a GPU this KV cache lives in HBM and can be **paged** (PagedAttention). On Spyre there is no HBM.
   Where would the KV cache have to live instead, and what new decision does that force on the compiler?

> **Your short answers:** Replace this text with your responses.


### 5b; KV Cache Formula Derivation + Spyre Scratchpad Sizing

The KV cache size formula:
```
KV_bytes = 2 × n_layers × n_heads × head_dim × seq_len × batch × bytes_per_element
```

**Why the formula works:** Each layer stores one K and one V tensor (hence the leading 2). Each has shape `[batch, n_heads, seq_len, head_dim]`. At FP16 (2 bytes per element), the total is `2 × layers × heads × head_dim × seq_len × batch × 2`.

**Spyre scratchpad constraint:** With ~2 MB per tile scratchpad and 32 tiles, the total on-chip capacity is ~64 MB. After reserving space for weights and activations of the current layer, the available space for KV cache per tile determines the maximum sequence length that can be served without streaming from LPDDR5 mid-computation.

In [ ]:
# Part 5b: KV Cache Formula + Scratchpad Sizing
import numpy as np

# Model config for distilgpt2
n_layers = model.config.n_layer
n_heads = model.config.n_head
head_dim = model.config.n_embd // model.config.n_head
bytes_per_elem = 2  # FP16

def kv_cache_bytes(seq_len, batch=1):
    """Compute KV cache size in bytes."""
    return 2 * n_layers * n_heads * head_dim * seq_len * batch * bytes_per_elem

# Print KV cache sizes at various sequence lengths
print('KV Cache Size by Sequence Length (batch=1, FP16):')
print(f'  {"Seq Len":>8} | {"KV Cache":>12} | {"Per Layer":>12}')
print(f'  {"-"*8} | {"-"*12} | {"-"*12}')
for seq in [128, 512, 1024, 2048, 4096, 8192]:
    total = kv_cache_bytes(seq)
    per_layer = total / n_layers
    print(f'  {seq:>8} | {total/1e6:>9.2f} MB | {per_layer/1e6:>9.4f} MB')

# Spyre scratchpad analysis
scratchpad_per_tile_mb = 2.0  # MB
n_tiles = 32
total_scratchpad_mb = scratchpad_per_tile_mb * n_tiles

# Assume ~50% of scratchpad available for KV after weights + activations
available_for_kv_mb = total_scratchpad_mb * 0.5
available_for_kv_bytes = available_for_kv_mb * 1e6

# Maximum sequence length that fits
# Solve: 2 * layers * heads * head_dim * seq * batch * bytes = available
max_seq = available_for_kv_bytes / (2 * n_layers * n_heads * head_dim * 1 * bytes_per_elem)
print(f'\nSpyre Scratchpad Analysis (distilgpt2):')
print(f'  Total scratchpad: {total_scratchpad_mb:.0f} MB')
print(f'  Available for KV (~50%): {available_for_kv_mb:.0f} MB')
print(f'  Max seq_len fitting in scratchpad: {int(max_seq):,} tokens')
print(f'  → Beyond this, KV cache must stream from LPDDR5')


### KV-Cache Size: MHA vs GQA vs MQA

The baseline KV-cache formula assumes **Multi-Head Attention (MHA)** where `n_kv_heads == n_heads`.
Modern architectures use fewer KV heads to reduce memory:

| Attention Type | KV Heads | Formula (per token, both K and V) | Example (32 heads, d=128, FP16) |
|---|---|---|---|
| **MHA** | n_heads | `2 × n_layers × n_heads × d_head × 2B` | 2 × L × 32 × 128 × 2 = 16,384L bytes |
| **GQA** (e.g. Llama-2 70B) | n_kv_groups | `2 × n_layers × n_kv_heads × d_head × 2B` | 2 × L × 8 × 128 × 2 = 4,096L bytes |
| **MQA** (e.g. PaLM) | 1 | `2 × n_layers × 1 × d_head × 2B` | 2 × L × 1 × 128 × 2 = 512L bytes |

> **Key insight:** GQA with 8 KV heads uses 4× less KV-cache memory than MHA with 32 heads,
> enabling longer context or larger batch sizes within the same memory budget.

In [ ]:
# KV-cache size calculator for MHA, GQA, and MQA
def kv_cache_bytes(n_layers, n_kv_heads, d_head, seq_len, dtype_bytes=2):
    """Compute KV-cache size in bytes for a single sequence.

    Args:
        n_layers: number of transformer layers
        n_kv_heads: number of KV heads (= n_heads for MHA, < n_heads for GQA, = 1 for MQA)
        d_head: dimension per head
        seq_len: sequence length (number of cached tokens)
        dtype_bytes: bytes per parameter (2 for FP16/BF16, 1 for INT8)
    Returns:
        Total KV-cache bytes (both K and V)
    """
    return 2 * n_layers * n_kv_heads * d_head * seq_len * dtype_bytes

# Example: Llama-2 style model (32 layers, 32 query heads, d_head=128)
n_layers, n_heads, d_head, seq_len = 32, 32, 128, 2048

mha_bytes = kv_cache_bytes(n_layers, n_heads, d_head, seq_len)         # MHA: all 32 heads
gqa_bytes = kv_cache_bytes(n_layers, 8, d_head, seq_len)               # GQA: 8 KV heads
mqa_bytes = kv_cache_bytes(n_layers, 1, d_head, seq_len)               # MQA: 1 KV head

print(f"KV-cache for seq_len={seq_len}, {n_layers} layers, d_head={d_head}, FP16:")
print(f"  MHA ({n_heads} KV heads): {mha_bytes / 1e6:.1f} MB")
print(f"  GQA (8 KV heads):  {gqa_bytes / 1e6:.1f} MB  ({mha_bytes/gqa_bytes:.0f}\u00d7 smaller)")
print(f"  MQA (1 KV head):   {mqa_bytes / 1e6:.1f} MB  ({mha_bytes/mqa_bytes:.0f}\u00d7 smaller)")

# Unit tests
assert mha_bytes == 2 * 32 * 32 * 128 * 2048 * 2
assert gqa_bytes == 2 * 32 * 8 * 128 * 2048 * 2
assert mqa_bytes == 2 * 32 * 1 * 128 * 2048 * 2
assert mha_bytes / gqa_bytes == 4.0
assert mha_bytes / mqa_bytes == 32.0
print("\n\u2713 All KV-cache unit tests passed")

---
## 6 · Exercise: Bottleneck annotation + analysis
For each of your top **three** memory bottlenecks, fill the table below (edit the markdown):

| Operator | Why it's memory-heavy | GPU cache behavior | Spyre scratchpad prediction | Better on? |
|---|---|---|---|---|
| _e.g. matmul_ | _re-reads weights each token_ | _cache thrashes at long context_ | _explicit bank placement keeps it ring-local_ | _Spyre / GPU / depends_ |
| | | | | |
| | | | | |

Then write a **250–400 word analysis**. A strong answer:
- Names the **specific operators** that dominated your memory profile and why (tie to arithmetic intensity).
- For each, predicts whether **scratchpad placement** (Spyre) or **cache-based movement** (GPU) helps, and
  justifies it with a *concrete* memory-hierarchy reason from this module (paging, NUMA-like ring locality,
  stick alignment, contiguous vs. fragmented).
- States one bottleneck where you expect **no** improvement from a different hierarchy, and why.

> **Your analysis:** Replace this text with your 250–400 word response.


---

## Part 6 - Update the Accelerator Landscape Map

Add this module's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row: the memory hierarchy you profiled this module.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Part 6 -- Landscape Map row: the memory hierarchy you profiled this module.
landscape_row_w4 = {
    "accelerator":      (f"GPU ({torch.cuda.get_device_name(0)})" if DEV == 'cuda' else "CPU"),
    "memory_hierarchy": "L1/L2 + GDDR/HBM (hardware-managed cache)",
    "peak_alloc_MB":    (round(torch.cuda.max_memory_allocated()/1e6, 1) if DEV == 'cuda' else None),
    "kv_cache_growth":  "linear in sequence length (Part 5 table)",
    "top_mem_operator": (top.iloc[0]['op'] if len(top) else None),
    "spyre_prediction": "explicit scratchpad placement; KV streamed under compiler control (no HBM paging)",
}
for _k, _v in landscape_row_w4.items():
    print(f"  {_k:18s}: {_v}")
```

</details>


---
## ★ Optional: Dataflow Deep Dive (Spyre hardware)
_Requires Spyre access via an IBM Research partnership. Skip on the core track._

1. Run the same generation on Spyre.
2. Use **hardware performance counters** to capture real memory traffic between scratchpad and DDR.
3. Compare the measured traffic to your GPU-profiler estimates from Sections 3–4.
4. Identify where **explicit scratchpad placement** measurably changes the memory pattern versus the GPU;    and at least one operator where it does **not**: and explain each in terms of the hierarchy.

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.


In [ ]:
# Spyre track only, pseudocode skeleton; consult your partnered-program setup guide.
# import torch_sendnn          # registers the 'aiu'/'spyre' device
# spyre = 'aiu'
# model_s = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(spyre).eval()
# ids_s = tok(PROMPT, return_tensors='pt').input_ids.to(spyre)
# counters = capture_hw_counters(lambda: model_s.generate(ids_s, max_new_tokens=48))
# compare(counters, top)   # scratchpad<->DDR traffic vs GPU estimate

<details>
<summary><strong>Hint</strong></summary>

| Operator | Why Memory-Heavy | GPU Cache Behavior | Spyre Prediction | Better On |
|----------|-----------------|-------------------|-----------------|----------|
| Embedding lookup | Random access into large table, low reuse | Cache misses likely for large vocab | Scratchpad too small for full table; must stream | GPU (cache handles random access) |
| QKV Projection (matmul) | Large weight read per token | Weight reuse across batch helps cache | Tiled; weights prefetched from LPDDR5 | Spyre if tiling is efficient |
| Attention (QK^T) | O(seq^2) intermediate tensor | Large intermediate may thrash L2 at long seq | Tiled attention avoids full materialization | Spyre (compiler tiles to fit) |
| LayerNorm | Reads + writes full activation, low compute | Good cache locality (sequential access) | Fused with adjacent ops to avoid round-trip | Spyre (fusion eliminates writes) |
| FFN (2 matmuls) | Largest weight matrices in the model | Weight reuse across batch | Tiled like QKV; benefits from double-buffering | Comparable |

**Key insight:** GPU excels at irregular access patterns (embedding) due to hardware caching. Spyre excels at regular, tileable workloads (matmuls, attention) where the compiler can plan perfect data movement. LayerNorm and residual adds benefit from fusion on both, but Spyre's compiler can fuse more aggressively because it controls the full memory schedule.

</details>
